# 📊 Step 7: DBSQL Dashboards — SOC Manager Analytics

This notebook contains the **analytical SQL queries** that power the SOC manager
dashboards. These queries run on **DBSQL warehouses** (not Lakebase) because
they perform full-table scans and aggregations.

### Dashboard Sections:
1. Alert Volume & Trends
2. MTTR (Mean Time to Respond) Tracking
3. False Positive Analysis
4. Analyst Performance
5. Customer Risk Overview

### Published AI/BI Dashboard
**SOC Alert Triage — Manager Overview** (ID `01f1c1048ec61c9591864deba1378da6`) — 2 pages, 19 widgets,
global filters for detection source and event date. Definition is version-controlled in
[`dashboards/soc_alert_triage.lvdash.json`](../dashboards/soc_alert_triage.lvdash.json).

The queries below are the analytical building blocks, run here on the SQL warehouse so their results
are captured as notebook output.

In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
spark.sql(f"""
    SELECT 
        event_date,
        source_type,
        risk_tier,
        COUNT(*) as alert_count,
        SUM(CASE WHEN threat_intel_match THEN 1 ELSE 0 END) as threat_intel_hits,
        ROUND(AVG(composite_risk_score), 2) as avg_risk_score
    FROM {TABLES['enriched_alerts']}
    GROUP BY event_date, source_type, risk_tier
    ORDER BY event_date, source_type
""").display()

event_date,source_type,risk_tier,alert_count,threat_intel_hits,avg_risk_score
2026-09-01,auth,medium,2048,0,3.84
2026-09-01,auth,high,224,31,5.43
2026-09-01,auth,critical,176,176,7.97
2026-09-01,auth,low,1770,0,2.14
2026-09-01,cloud_audit,low,569,0,2.17
2026-09-01,cloud_audit,medium,923,0,3.92
2026-09-01,cloud_audit,high,192,0,5.39
2026-09-01,edr,low,655,0,2.44
2026-09-01,edr,medium,2049,0,3.98
2026-09-01,edr,high,707,0,5.48


In [0]:
spark.sql(f"""
    SELECT 
        event_date,
        COUNT(*) as total_alerts,
        SUM(CASE WHEN risk_tier = 'critical' THEN 1 ELSE 0 END) as critical,
        SUM(CASE WHEN risk_tier = 'high' THEN 1 ELSE 0 END) as high,
        SUM(CASE WHEN risk_tier = 'medium' THEN 1 ELSE 0 END) as medium,
        SUM(CASE WHEN risk_tier = 'low' THEN 1 ELSE 0 END) as low,
        ROUND(SUM(CASE WHEN threat_intel_match THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as threat_intel_match_pct
    FROM {TABLES['enriched_alerts']}
    GROUP BY event_date
    ORDER BY event_date
""").display()

event_date,total_alerts,critical,high,medium,low,threat_intel_match_pct
2026-09-01,16991,435,2357,8597,5602,2.98
2026-09-02,16860,399,2350,8537,5574,2.80
2026-09-03,16730,386,2358,8403,5583,2.74
2026-09-04,16704,406,2364,8421,5513,2.91
2026-09-05,16462,376,2519,8445,5122,2.72
2026-09-06,16547,405,2613,8351,5178,2.89
2026-09-07,16634,401,2382,8374,5477,2.88
2026-09-08,16699,401,2266,8522,5510,2.78
2026-09-09,16787,392,2444,8487,5464,2.90
2026-09-10,16661,382,2333,8411,5535,2.71


In [0]:
spark.sql(f"""
    SELECT 
        analyst_id,
        COUNT(*) as total_actions,
        ROUND(AVG(time_to_triage_seconds) / 60, 1) as avg_triage_minutes,
        ROUND(PERCENTILE(time_to_triage_seconds, 0.5) / 60, 1) as median_triage_minutes,
        ROUND(PERCENTILE(time_to_triage_seconds, 0.95) / 60, 1) as p95_triage_minutes,
        SUM(CASE WHEN action_type = 'escalate' THEN 1 ELSE 0 END) as escalations,
        SUM(CASE WHEN action_type = 'false_positive' THEN 1 ELSE 0 END) as false_positives
    FROM {TABLES['analyst_actions']}
    GROUP BY analyst_id
    ORDER BY avg_triage_minutes
""").display()

analyst_id,total_actions,avg_triage_minutes,median_triage_minutes,p95_triage_minutes,escalations,false_positives
analyst_15,59,17.7,15.7,40.7,16,7
analyst_1,34,19.4,19.9,40.1,4,8
analyst_7,38,21.4,20.1,41.4,4,3
analyst_10,61,22.1,19.9,43.0,14,12
analyst_2,39,22.2,21.2,39.8,7,6
analyst_20,51,22.3,21.6,43.4,15,14
analyst_13,57,22.6,22.1,42.9,10,12
analyst_6,46,22.6,24.5,42.9,9,10
analyst_4,51,22.6,20.1,44.0,13,13
analyst_18,56,22.8,22.8,44.2,14,7


In [0]:
spark.sql(f"""
    SELECT 
        source_type,
        COUNT(*) as total_triage,
        SUM(CASE WHEN triage_outcome = 'false_positive' THEN 1 ELSE 0 END) as false_positives,
        ROUND(SUM(CASE WHEN triage_outcome = 'false_positive' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as fp_rate_pct,
        SUM(CASE WHEN triage_outcome = 'true_positive' THEN 1 ELSE 0 END) as true_positives,
        ROUND(AVG(triage_time_minutes), 1) as avg_triage_minutes
    FROM {TABLES['historical_triage']}
    GROUP BY source_type
    ORDER BY fp_rate_pct DESC
""").display()

source_type,total_triage,false_positives,fp_rate_pct,true_positives,avg_triage_minutes
cloud_audit,10012,4568,45.63,1498,27.7
edr,10039,4575,45.57,1542,27.9
auth,10081,4579,45.42,1478,27.8
firewall,9970,4526,45.40,1473,27.8
email,9898,4453,44.99,1473,27.7


In [0]:
spark.sql(f"""
    SELECT 
        customer_name,
        industry,
        tier,
        region,
        COUNT(*) as total_alerts,
        SUM(CASE WHEN risk_tier IN ('critical', 'high') THEN 1 ELSE 0 END) as high_risk_alerts,
        ROUND(AVG(composite_risk_score), 2) as avg_risk_score,
        SUM(CASE WHEN threat_intel_match THEN 1 ELSE 0 END) as threat_intel_matches
    FROM {TABLES['enriched_alerts']}
    GROUP BY customer_name, industry, tier, region
    ORDER BY high_risk_alerts DESC
    LIMIT 20
""").display()

customer_name,industry,tier,region,total_alerts,high_risk_alerts,avg_risk_score,threat_intel_matches
Horizon Systems,Energy,Mid-Market,US-West,4995,1113,3.9,142
Acme LLC,Retail,Enterprise,EMEA,4964,1032,3.83,150
Horizon Industries,Education,Enterprise,APAC,5010,1013,3.78,150
Acme Technologies,Energy,Enterprise,US-West,4945,1001,3.81,132
Global Partners,Logistics,Enterprise,US-Central,4978,999,3.8,152
Apex LLC,Retail,SMB,LATAM,4961,988,3.75,120
Vertex LLC,Retail,Mid-Market,EMEA,5089,984,3.77,145
Acme Industries,Healthcare,Enterprise,APAC,5053,979,3.77,142
Pacific Inc,Energy,Mid-Market,EMEA,5041,973,3.77,126
Atlantic Inc,Logistics,Enterprise,APAC,4984,957,3.8,114


In [0]:
spark.sql(f"""
    SELECT 
        mitre_tactic,
        risk_tier,
        COUNT(*) as alert_count,
        ROUND(AVG(composite_risk_score), 2) as avg_risk
    FROM {TABLES['enriched_alerts']}
    WHERE source_type = 'edr' AND mitre_tactic IS NOT NULL
    GROUP BY mitre_tactic, risk_tier
    ORDER BY alert_count DESC
""").display()

mitre_tactic,risk_tier,alert_count,avg_risk
Execution,medium,6086,3.98
Persistence,medium,5974,3.98
Lateral Movement,medium,5923,3.99
Credential Access,medium,5907,3.98
Privilege Escalation,medium,5906,3.99
Initial Access,medium,5904,3.99
Discovery,medium,5896,3.98
Collection,medium,5860,3.99
Defense Evasion,medium,5858,3.99
Exfiltration,medium,5843,3.99


In [0]:
spark.sql(f"""
    SELECT 
        event_hour as hour,
        CASE dayofweek(event_date)
            WHEN 1 THEN 'Sun' WHEN 2 THEN 'Mon' WHEN 3 THEN 'Tue'
            WHEN 4 THEN 'Wed' WHEN 5 THEN 'Thu' WHEN 6 THEN 'Fri'
            WHEN 7 THEN 'Sat'
        END as day_of_week,
        dayofweek(event_date) as day_num,
        COUNT(*) as alert_count,
        SUM(CASE WHEN risk_tier IN ('critical', 'high') THEN 1 ELSE 0 END) as high_risk_count
    FROM {TABLES['enriched_alerts']}
    GROUP BY event_hour, dayofweek(event_date), 
        CASE dayofweek(event_date)
            WHEN 1 THEN 'Sun' WHEN 2 THEN 'Mon' WHEN 3 THEN 'Tue'
            WHEN 4 THEN 'Wed' WHEN 5 THEN 'Thu' WHEN 6 THEN 'Fri'
            WHEN 7 THEN 'Sat'
        END
    ORDER BY day_num, hour
""").display()

hour,day_of_week,day_num,alert_count,high_risk_count
0,Sun,1,2833,502
1,Sun,1,2735,501
2,Sun,1,2741,507
3,Sun,1,2809,491
4,Sun,1,2806,519
5,Sun,1,2800,540
6,Sun,1,2746,480
7,Sun,1,2817,488
8,Sun,1,2757,467
9,Sun,1,2792,498


In [0]:
spark.sql(f"""
    SELECT 
        p.ml_predicted_severity,
        COUNT(*) as prediction_count,
        ROUND(AVG(p.ml_confidence), 4) as avg_confidence,
        ROUND(AVG(e.composite_risk_score), 2) as avg_actual_risk,
        SUM(CASE WHEN e.threat_intel_match THEN 1 ELSE 0 END) as threat_matches
    FROM {TABLES['alert_predictions']} p
    JOIN {TABLES['enriched_alerts']} e ON p.event_id = e.event_id
    GROUP BY p.ml_predicted_severity
    ORDER BY avg_actual_risk DESC
""").display()

ml_predicted_severity,prediction_count,avg_confidence,avg_actual_risk,threat_matches
critical,49798,0.7907,6.17,13642
high,95260,0.5862,4.79,77
medium,118059,0.5684,3.97,0
low,117365,0.5775,3.06,2
informational,119518,0.8154,2.07,0


## ✅ DBSQL Dashboard Queries Complete!

### Dashboard contents
- **Overview**: KPIs (total/critical alerts, % of real threats the ML model ranked critical/high,
  % of alerts ML ranks low/informational, avg triage time), daily trend by risk tier, ML severity vs rule-based tier,
  real threats vs ML-prioritized by source, top 10 customers, EDR alerts by MITRE tactic
- **Customers & Analysts**: customer risk table, analyst performance (from app actions), FP rate by source

**Next step**: Set up the Genie space with `08_genie_agent_setup`